In [1]:
#Imports
import os
import requests
from dotenv import load_dotenv
from bs4 import BeautifulSoup
from IPython.display import Markdown, display
from openai import OpenAI

In [2]:
#Load environment variables i.e. api key in this case
load_dotenv(override = True)
api_key = os.getenv('OPENAI_API_KEY')

In [3]:
#Ensure api key is a valid key
if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [4]:
#Instantiate OpenAI. Remember this automatically picks api key from the environment
openai = OpenAI()

In [5]:
#Test the model
message = "Hello, GPT! This is my first ever message to you! Hi!"
response = openai.chat.completions.create(model = "gpt-4o-mini", messages = [{"role": "user", "content": message}])
print(response.choices[0].message.content)

Hello! Welcome! I'm glad you're here and excited to chat with you. How can I assist you today?


In [6]:
#Configure headers
headers = {
 "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

In [7]:
#Define a class
class Website:
    def __init__(self, url):
        self.url = url
        response = requests.get(url, headers = headers)
        soup = BeautifulSoup(response.content, "html.parser")
        self.title = soup.title.string if soup.title else "No title found"
        
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        
        self.text = soup.body.get_text(separator = "\n", strip = True)

In [8]:
#Test website contents are getting fetched or not
ed = Website("https://cnn.com")
print(ed.title)
print(ed.text)

Breaking News, Latest News and Videos | CNN
CNN values your feedback
1. How relevant is this ad to you?
2. Did you encounter any technical issues?
Video player was slow to load content
Video content never loaded
Ad froze or did not finish loading
Video content did not start after ad
Audio on ad was too loud
Other issues
Ad never loaded
Ad prevented/slowed the page from loading
Content moved around while ad loaded
Ad was repetitive to ads I've seen previously
Other issues
Cancel
Submit
Thank You!
Your effort and contribution in providing this feedback is much
                                        appreciated.
Close
Ad Feedback
Close icon
US
World
Politics
Business
Health
Entertainment
Style
Travel
Sports
Science
Climate
Weather
Ukraine-Russia War
Israel-Hamas War
Games
More
US
World
Politics
Business
Health
Entertainment
Style
Travel
Sports
Science
Climate
Weather
Ukraine-Russia War
Israel-Hamas War
Games
Watch
Listen
Live TV
Subscribe
Sign in
My Account
Settings
Newsletters
Topics yo

In [9]:
#Define system prompt for better context setting
system_prompt = "You are an assistant that analyzes the contents of a website \
and provides a short summary, ignoring text that might be navigation related. \
Respond in markdown."

In [10]:
#A function that writes a User Prompt that asks for summaries of websites
def user_prompt_for(website):
    user_prompt = f"You are looking at a website titled {website.title}"
    user_prompt += "\nThe contents of this website is as follows; \
please provide a short summary of this website in markdown. \
If it includes news or announcements, then summarize these too.\n\n"
    user_prompt += website.text
    return user_prompt

In [11]:
#A function which constructs message for our summarizer
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_for(website)}
    ]

In [12]:
#Main summarizer function
def summarize(url):
    website = Website(url)
    response = openai.chat.completions.create(
        model = "gpt-4o-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [13]:
#Function to show summary in markdown style
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [14]:
display_summary("https://cnn.com")

# CNN Website Summary

**CNN** is a major news network that provides up-to-date reporting on a wide range of topics including US and world news, politics, business, health, entertainment, science, and sports. The website features various types of content, including articles, video reports, and interactive features.

## Key Sections:
- **Top News**: Highlights of significant global events, including ongoing conflicts like the Ukraine-Russia war and the Israel-Hamas situation.
- **Politics**: Coverage of important political developments, including potential implications of a US government shutdown and controversies surrounding figures like Trump.
- **Health & Wellness**: Personal essays and discussions on wellness and health-related topics.
- **Business & Technology**: Analysis on market trends, technology innovations, and major corporate developments.
- **Entertainment**: Updates from the entertainment industry, including interviews and trends regarding popular culture.
- **Science & Climate**: Reports on scientific discoveries and updates on climate initiatives.

## Recent Highlights:
1. **Breaking News**: Continuous updates on political discourse, including Trump’s position on birthright citizenship and significant legislative actions.
2. **Critical Global Issues**: Reports discuss the impact of Russia's actions in Europe and recent developments in negotiations involving key figures like Hamas negotiators.
3. **Personal Perspectives**: Features explore personal insights on broader societal issues, such as commentary on health misconceptions and lifestyle choices.

CNN encourages user engagement and feedback regarding their experience on the site, particularly in relation to ad relevance and technical performance. The platform also includes interactive content such as games and quizzes to engage users.

Overall, CNN serves as a comprehensive source for both breaking news and in-depth analysis across multiple topics of interest.